# Prétraitement et Extraction de Features

Ce notebook vise à transformer les images PNG/JPG en vecteurs numériques exploitables via `ResNet50` pré-entraîné sur `ImageNet`.

In [1]:
# initialisation du modèle ResNet50 pré-entraîné pour l'extraction de caractéristiques.

import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"🖥️  Utilisation de l'accélération matérielle: {device}")

🖥️  Utilisation de l'accélération matérielle: mps


In [2]:
# Boucle d'extraction de caractéristiques sur l'ensemble du dataset (labellisé et non labellisé).
# Configuration de l'extraction (Recommendation du skill : évaluation de la richesse des couches)
# Options: 'layer4' (GAP - 2048 dims), 'layer3' (GAP - 1024 dims)
EXTRACTION_LAYER = 'layer4' 
BATCH_SIZE = 32

print(f"⏳ Chargement du modèle ResNet50 (Mode: {EXTRACTION_LAYER})...")
weights = ResNet50_Weights.IMAGENET1K_V1
base_model = models.resnet50(weights=weights)

if EXTRACTION_LAYER == 'layer4':
    # Jusqu'à l'AvgPool final (2048 dims)
    model = nn.Sequential(*list(base_model.children())[:-1])
elif EXTRACTION_LAYER == 'layer3':
    # Jusqu'à Layer 3 + Pooling adaptatif (1024 dims)
    model = nn.Sequential(
        *list(base_model.children())[:-3],
        nn.AdaptiveAvgPool2d((1, 1))
    )

model = model.to(device)
model.eval()

# Geler les couches convolutives
for param in model.parameters():
    param.requires_grad = False

⏳ Chargement du modèle ResNet50 (Mode: layer4)...


In [3]:

# Transformation pipeline (Redimensionnement et Normalisation ImageNet)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
# Dataset personnalisé pour charger les images MRI et leurs labels à partir des dossiers spécifiés.
class MRIDataset(Dataset):
    def __init__(self, folders, transform=None):
        """
        @definition : Initialisation du dataset MRI avec chargement des chemins d'images et labels.
        @args/params : folders (dict), transform (callable)
        @return : None
        """
        self.image_info = []
        self.transform = transform
        for label, folder in folders.items():
            if os.path.exists(folder):
                paths = glob.glob(os.path.join(folder, "*.jpg")) + glob.glob(os.path.join(folder, "*.png"))
                for p in paths:
                    self.image_info.append((p, label))
    
    def __len__(self):
        """
        @definition : Retourne le nombre total d'images dans le dataset.
        @args/params : None
        @return : int
        """
        return len(self.image_info)
    
    def __getitem__(self, idx):
        """
        @definition : Récupère une image et son label par index.
        @args/params : idx (int)
        @return : tuple (image, label, filename)
        """
        img_path, label = self.image_info[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, os.path.basename(img_path)

base_dir = "../data/raw/mri_dataset_brain_cancer_oc"
folders = {
    "Normal": os.path.join(base_dir, "avec_labels", "Normal"),
    "Cancer": os.path.join(base_dir, "avec_labels", "Cancer"),
    "non_labellise": os.path.join(base_dir, "sans_label")
}

dataset = MRIDataset(folders, transform=preprocess)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

extracted_features = []
extracted_filenames = []
extracted_labels = []

print(f"Début de l'extraction sur {len(dataset)} images...")
for imgs, labels, fnames in tqdm(dataloader, desc="Extraction des features"):
    imgs = imgs.to(device)
    with torch.no_grad():
        features = model(imgs)
    
    features = features.view(features.size(0), -1).cpu().numpy()
    
    extracted_features.extend(features)
    extracted_filenames.extend(fnames)
    extracted_labels.extend(labels)

Début de l'extraction sur 1506 images...


Extraction des features:   0%|          | 0/48 [00:00<?, ?it/s]

In [5]:
# Sauvegarde dans un fichier performant compressé (.npy)
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "features_brainscan.npy")

features_array = np.array(extracted_features)
filenames_array = np.array(extracted_filenames)
labels_array = np.array(extracted_labels)

if len(features_array) > 0:
    np.save(output_path, {
        "features": features_array,
        "filenames": filenames_array,
        "labels": labels_array
    }, allow_pickle=True)
    print(f"✅ {len(features_array)} embeddings sauvegardés avec succès.")
    print(f"Shape des features : {features_array.shape}")
else:
    print("⚠️ Aucune feature extraite.")

✅ 1506 embeddings sauvegardés avec succès.
Shape des features : (1506, 2048)
